In [ ]:
#!pip install langchain transformers pypdf faiss-cpu sentence-transformers
#!pip install langchain_community
#!pip install langchain_huggingface

In [8]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS

# Load and process the PDF
def build_and_save_faiss_store(pdf_path, store_path="faiss_store"):
    # Load the PDF into LangChain documents
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    # Split documents into smaller chunks
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    split_docs = text_splitter.split_documents(documents)

    # Create FAISS vector store
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vector_store = FAISS.from_documents(split_docs, embeddings)

    # Save the store
    vector_store.save_local(store_path)
    print(f"FAISS store saved to {store_path}!")

# Example: Build and save FAISS store
pdf_path = "example.pdf"  # Replace with your PDF file
build_and_save_faiss_store(pdf_path)


FAISS store saved to faiss_store!


In [10]:
from langchain.vectorstores import FAISS
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Load the FAISS store
def load_faiss_store(store_path="faiss_store"):
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vector_store = FAISS.load_local(store_path, embeddings,allow_dangerous_deserialization=True)
    return vector_store

# Initialize GPT-2 for generation
def setup_gpt2_pipeline():
    model_name = "gpt2"
    tokenizer = GPT2Tokenizer.from_pretrained(model_name)
    model = GPT2LMHeadModel.from_pretrained(model_name)
    return tokenizer, model

# Example: Load the store and GPT-2
vector_store = load_faiss_store()
tokenizer, gpt2_model = setup_gpt2_pipeline()


In [11]:
import torch

def rag_with_gpt2(query, vector_store, tokenizer, gpt2_model):
    # Step 1: Retrieve relevant chunks
    retriever = vector_store.as_retriever()
    relevant_docs = retriever.get_relevant_documents(query)
    context = " ".join([doc.page_content for doc in relevant_docs])

    # Step 2: Prepare input for GPT-2
    input_text = f"Context: {context}\n\nQuestion: {query}\n\nAnswer:"
    inputs = tokenizer.encode(input_text, return_tensors="pt", truncation=True, max_length=1024)

    # Set attention mask
    attention_mask = torch.ones(inputs.shape, dtype=torch.long)


    outputs = gpt2_model.generate(
        inputs,
        attention_mask=attention_mask,
        max_new_tokens=200,
        num_beams=3,
        early_stopping=True,
        pad_token_id=tokenizer.eos_token_id
    )
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer

# Example: Query with RAG
query = "What is the document about?"
response = rag_with_gpt2(query, vector_store, tokenizer, gpt2_model)
print("\n--- Generated Response ---")
print(response)

C:\Users\wayne\AppData\Local\Temp\ipykernel_59720\4269839126.py:6: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  relevant_docs = retriever.get_relevant_documents(query)



--- Generated Response ---
Context: advisory council, which first met in December 2022.
The Kinship and Prosperity Report focuses on six key themes: easing access to
funding; developing consistent project eligibility criteria that prioritize Indigenous
community benefits; advancing inclusive opportunities and a Just Transition;
accelerating Indigenous leadership in the energy transition; respecting self-
determination by prioritizing Indigenous-led decisions; and sustainably funding
Indigenous participation.
1.3 Key Guiding Principles
Six key principles underpin the Clean Electricity Strategy and will guide federal
action to support electricity grid decarbonization and expansion.
Principle 1: Provincial and Territorial Jurisdiction Must be Respected and
Supported with Policy Certainty
Provinces and territories hold jurisdiction over electricity planning and operations
within their borders, while the federal government regulates nuclear energy and five years or less, which will apply t

In [12]:
# Get all documents from the vector store
documents = []
for i in range(len(vector_store.index_to_docstore_id)):
    doc_id = vector_store.index_to_docstore_id[i]
    doc = vector_store.docstore.search(doc_id)
    documents.append(doc)

# Print summary info
print(f"Total documents: {len(documents)}")
print("\nSample of documents:")
for i, doc in enumerate(documents[:3]):  # Show first 3 docs
    print(f"\nDocument {i+1}:")
    print(f"ID: {doc.id}")
    print(f"Metadata: {doc.metadata}")
    print("Content preview (first 200 chars):")
    print(doc.page_content[:200])
    print("-" * 80)

Total documents: 198

Sample of documents:

Document 1:
ID: 2a3a8887-bde2-4f0f-8e8e-1fb2509a971f
Metadata: {'producer': 'Skia/PDF m128', 'creator': 'Chromium', 'creationdate': '2025-01-22T20:42:49+00:00', 'title': 'Powering Canada’s Future: A Clean Electricity Strategy', 'moddate': '2025-01-22T20:42:49+00:00', 'source': 'example.pdf', 'total_pages': 61, 'page': 0, 'page_label': '1'}
Content preview (first 200 chars):
natural-resources.canada.ca /our-natural-resources/energy-sources-distribution/electricity-infrastru…
Powering Canada’s Future: A Clean Electricity
Strategy
155-197 minutes
Table of Contents
Foreword 
--------------------------------------------------------------------------------

Document 2:
ID: d3490d43-f4d1-419a-86d3-2eee00fcd1af
Metadata: {'producer': 'Skia/PDF m128', 'creator': 'Chromium', 'creationdate': '2025-01-22T20:42:49+00:00', 'title': 'Powering Canada’s Future: A Clean Electricity Strategy', 'moddate': '2025-01-22T20:42:49+00:00', 'source': 'example.pdf', 'to

In [13]:
# Get the actual vectors from FAISS index
vectors = vector_store.index.reconstruct_n(0, vector_store.index.ntotal)

print(f"Number of vectors: {len(vectors)}")
print(f"Vector dimension: {vectors[0].shape}")

# Look at first vector
print("\nFirst vector (first 10 dimensions):")
print(vectors[0][:10])

# Basic vector stats
print("\nVector statistics:")
print(f"Mean: {vectors.mean()}")
print(f"Min: {vectors.min()}")
print(f"Max: {vectors.max()}")

Number of vectors: 198
Vector dimension: (384,)

First vector (first 10 dimensions):
[-0.00191794  0.04656455  0.08821359  0.02866852  0.02933782 -0.03801762
 -0.02533459 -0.05188401 -0.03644834  0.03173688]

Vector statistics:
Mean: -0.0002986040199175477
Min: -0.20857585966587067
Max: 0.22076773643493652


In [14]:
# Perform similarity search
search_query = "What is the role of Indigenous communities in clean energy projects?"
similar_docs = vector_store.similarity_search(
    search_query,
    k=3  # get top 3 most similar chunks
)

# Print results
print(f"Search query: '{search_query}'\n")
print("Top similar documents:")
for i, doc in enumerate(similar_docs):
    print(f"\nDocument {i+1}:")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}, Page: {doc.metadata.get('page', 'Unknown')}")
    print("-" * 80)
    print(doc.page_content)
    print("-" * 80)

Search query: 'What is the role of Indigenous communities in clean energy projects?'

Top similar documents:

Document 1:
Source: example.pdf, Page: 11
--------------------------------------------------------------------------------
Indigenous Peoples are key leaders and partners in helping to transform the
electricity sector across Canada.
This includes the growing number of Indigenous communities and proponents that
are building projects to supply clean electricity and reduce reliance on fossil fuels.
As the final report of the Wah-ila-toos Indigenous Council makes clear, to realize the
full potential of Indigenous-led clean energy projects, Indigenous communities and
businesses must have the appropriate tools and capacity. This matters because
clean energy projects can provide tangible community benefits such as training, job
creation, and skills development, which empower local expertise and enhance
energy literacy. Over the long-term this will foster control over energy systems an